<a href="https://colab.research.google.com/github/AdriBell01/context-sensitive-design-of-a-medical-device-/blob/main/CicCore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
# CELLA 1 — Setup e installazione dipendenze
# =====================================================================
!pip install librosa scikit-learn tensorflow tqdm matplotlib numpy pandas -q

import os
import numpy as np
import pandas as pd
import librosa
import librosa.display
import matplotlib.pyplot as plt
import pickle
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.utils import class_weight
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
print("TensorFlow version:", tf.__version__)
print("Setup completato.")

In [ ]:
# =====================================================================
# CELLA 2 — Download dataset CirCor da PhysioNet
# =====================================================================
# Scarica il dataset direttamente da PhysioNet (richiede accettazione
# della licenza su https://physionet.org/content/circor-heart-sound/)
# Se hai già il dataset su Drive, monta Drive e salta questo blocco.

!wget -r -N -c -np \
  https://physionet.org/files/circor-heart-sound/1.0.3/ \
  -P /content/circor/ \
  --quiet

DATASET_PATH = '/content/circor/physionet.org/files/circor-heart-sound/1.0.3/training_data'
CSV_PATH     = '/content/circor/physionet.org/files/circor-heart-sound/1.0.3/training_data.csv'

# Verifica
df_meta = pd.read_csv(CSV_PATH)
print(f"Pazienti nel dataset: {len(df_meta)}")
print(df_meta.head())

In [ ]:
# =====================================================================
# CELLA 3 — Preprocessor CirCor
# =====================================================================
# Il dataset CirCor ha 3 label di murmur: Present / Absent / Unknown
# Le mappiamo su:
#   - "Absent"  → classe 0 (Normal)
#   - "Present" → classe 1 (Murmur / proxy RHD Subclinical)
#   - "Unknown" → scartato (qualità non garantita)
# Successivamente i dati sintetici RHD aggiungeranno la classe 2 (RHD Advanced)

LABEL_MAP = {
    'Absent':  0,  # Normal
    'Present': 1,  # Murmur (proxy RHD Subclinical)
    # 'Unknown' viene scartato
}

SR          = 4000   # Hz — sampling rate target
N_MELS      = 64     # numero di filtri Mel
N_FFT       = 256    # lunghezza FFT
HOP_LENGTH  = 64     # hop tra frame consecutivi
SG_MAXLEN   = 256    # lunghezza massima (in frame) di ogni segmento
PADDING_SEC = 0.2    # secondi di padding attorno a ogni ciclo cardiaco

class CircorProcessor:
    def __init__(self, csv_path, data_path, label_map,
                 sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                 hop_length=HOP_LENGTH, sg_maxlen=SG_MAXLEN,
                 padding_sec=PADDING_SEC):

        self.df         = pd.read_csv(csv_path)
        self.data_path  = data_path
        self.label_map  = label_map
        self.sr         = sr
        self.sg_settings = {
            'n_mels':     n_mels,
            'n_fft':      n_fft,
            'hop_length': hop_length
        }
        self.sg_maxlen  = sg_maxlen
        self.padding    = int(padding_sec * sr)

    # ------------------------------------------------------------------
    def process_all(self):
        """
        Itera su tutti i pazienti, restituisce:
          X: array (N, n_mels, sg_maxlen) di spettrogrammi
          y: array (N,) di etichette intere
        """
        X_list, y_list = [], []

        for _, row in tqdm(self.df.iterrows(), total=len(self.df)):
            patient_id = row['Patient ID']
            label_str  = row['Murmur']

            # Scarta Unknown
            if label_str not in self.label_map:
                continue
            label = self.label_map[label_str]

            # Leggi il file .txt che descrive le registrazioni del paziente
            txt_path = os.path.join(self.data_path, f'{patient_id}.txt')
            if not os.path.exists(txt_path):
                continue

            with open(txt_path, 'r') as f:
                lines = f.readlines()

            _, num_records, orig_sr = [int(s) for s in lines[0].strip().split(' ')]

            for n in range(num_records):
                parts = lines[n + 1].strip().split(' ')
                auscultation_point = parts[0]  # AV, TV, PV, MV
                wav_file           = parts[2]
                tsv_file           = parts[3]

                try:
                    batch = self._get_sg_batch(orig_sr, wav_file, tsv_file)
                    for sg in batch:
                        X_list.append(sg)
                        y_list.append(label)
                except Exception as e:
                    print(f"Errore su {tsv_file}: {e}")

        X = np.array(X_list, dtype=np.float32)
        y = np.array(y_list, dtype=np.int32)
        print(f"Dataset CirCor — shape X: {X.shape}, shape y: {y.shape}")
        print(f"Distribuzione classi: {dict(zip(*np.unique(y, return_counts=True)))}")
        return X, y

    # ------------------------------------------------------------------
    def _load_wav_as_melspec(self, wav_file):
        sig, _ = librosa.load(
            os.path.join(self.data_path, wav_file),
            sr=self.sr
        )
        S = librosa.feature.melspectrogram(
            y=sig, sr=self.sr, **self.sg_settings
        )
        return librosa.power_to_db(S, ref=np.max)

    # ------------------------------------------------------------------
    def _get_tsv_segments(self, orig_sr, tsv_file):
        """
        Legge il file .tsv e restituisce gli indici di inizio/fine
        di ogni ciclo cardiaco completo (1-2-3-4).
        """
        df_tsv = pd.read_csv(
            os.path.join(self.data_path, tsv_file),
            sep='\t', header=None
        )
        # Colonne: [start_sample, end_sample, phase (1/2/3/4)]
        df_tsv.iloc[:, 0:2] = df_tsv.iloc[:, 0:2] * orig_sr

        # Identifica sequenze complete 1-2-3-4
        phases = df_tsv.iloc[:, 2].astype(str).tolist()
        phase_str = ''.join(phases)
        good_mask = []
        i = 0
        while i < len(phases):
            if phase_str[i:i+4] == '1234':
                good_mask.extend([True, True, True, True])
                i += 4
            else:
                good_mask.append(False)
                i += 1

        df_tsv['good'] = good_mask
        starts = df_tsv[df_tsv.iloc[:, 2] == 1][df_tsv['good']].iloc[:, 0].astype(int).tolist()
        ends   = df_tsv[df_tsv.iloc[:, 2] == 4][df_tsv['good']].iloc[:, 1].astype(int).tolist()
        return starts, ends

    # ------------------------------------------------------------------
    def _get_sg_batch(self, orig_sr, wav_file, tsv_file):
        S_dB = self._load_wav_as_melspec(wav_file)
        starts, ends = self._get_tsv_segments(orig_sr, tsv_file)

        # Numero totale di sample nel file (ultima riga del tsv)
        df_tsv = pd.read_csv(
            os.path.join(self.data_path, tsv_file),
            sep='\t', header=None
        )
        num_samples = int(df_tsv.iloc[-1, 1] * orig_sr)
        scale = S_dB.shape[1] / num_samples

        batch = []
        for s, e in zip(starts, ends):
            s_scaled = max(0, int(s * scale) - self.padding)
            e_scaled = min(S_dB.shape[1], int(e * scale) + self.padding)

            sg = S_dB[:, s_scaled:e_scaled]

            # Porta tutti i segmenti alla stessa lunghezza (padding o crop)
            pad_width = self.sg_maxlen - sg.shape[1]
            if pad_width > 0:
                sg = np.pad(sg, ((0, 0), (0, pad_width)),
                            mode='constant', constant_values=S_dB.min())
            else:
                sg = sg[:, :self.sg_maxlen]

            batch.append(sg)

        return batch  # lista di array (n_mels, sg_maxlen)


# Esegui il processing
processor = CircorProcessor(CSV_PATH, DATASET_PATH, LABEL_MAP)
X_circor, y_circor = processor.process_all()

# Salva per non doverlo rifare
with open('/content/circor_data.pkl', 'wb') as f:
    pickle.dump({'X': X_circor, 'y': y_circor}, f)
print("CirCor salvato in /content/circor_data.pkl")

In [ ]:
# =====================================================================
# CELLA 4 — Generazione dati sintetici RHD Advanced (classe 2)
# =====================================================================
# Genera segnali PCG sintetici che simulano le caratteristiche acustiche
# della RHD avanzata (rigurgito mitralico severo):
#   - Soffio olosistolico ad alta intensità (20–600 Hz)
#   - S1 attenuato (fibrosi valvolare riduce il tono di chiusura)
#   - S2 preservato o sdoppiato
#   - Rumore di fondo ambientale (simula condizioni di campo)

def generate_rhd_pcg(
    sr          = SR,
    duration    = 3.0,    # secondi
    heart_rate  = 90,     # bpm (bambini 5-15 anni)
    murmur_intensity = 0.6,
    noise_level = 0.05,
    seed        = None
):
    """
    Genera un segnale PCG sintetico con caratteristiche di RHD avanzata.
    Restituisce un array numpy (n_samples,).
    """
    if seed is not None:
        np.random.seed(seed)

    n_samples   = int(sr * duration)
    t           = np.linspace(0, duration, n_samples)
    signal      = np.zeros(n_samples)

    # Periodo cardiaco in secondi
    rr_interval = 60.0 / heart_rate

    # Posizioni dei battiti nel tempo
    beat_times = np.arange(0, duration, rr_interval)

    for bt in beat_times:
        # --- S1 (chiusura mitrale/tricuspide) — attenuato in RHD ---
        s1_center = bt
        s1_amp    = 0.4 + np.random.uniform(-0.05, 0.05)  # attenuato
        s1_dur    = 0.08  # secondi
        s1_freq   = 80    # Hz
        s1_mask   = (t >= s1_center) & (t < s1_center + s1_dur)
        env_s1    = np.exp(-((t[s1_mask] - s1_center) / (s1_dur / 3)) ** 2)
        signal[s1_mask] += s1_amp * env_s1 * np.sin(2 * np.pi * s1_freq * t[s1_mask])

        # --- Soffio olosistolico (da rigurgito mitralico) ---
        # Occupa tutta la sistole: da S1 a S2
        systole_start = s1_center + 0.02
        systole_end   = s1_center + rr_interval * 0.42  # ~42% RR = sistole
        murmur_mask   = (t >= systole_start) & (t < systole_end)

        if murmur_mask.sum() > 0:
            # Componente broadband (rumore di turbolenza valvolare)
            broadband = np.random.randn(murmur_mask.sum())
            # Componente tonica a frequenza caratteristica (100-400 Hz)
            f_murmur  = np.random.uniform(150, 350)
            tonal     = np.sin(2 * np.pi * f_murmur * t[murmur_mask])
            # Envelope: plateau con onset/offset smussati
            n_murmur  = murmur_mask.sum()
            ramp      = int(0.05 * sr)  # 50 ms di rampa
            envelope  = np.ones(n_murmur)
            if n_murmur > 2 * ramp:
                envelope[:ramp]  = np.linspace(0, 1, ramp)
                envelope[-ramp:] = np.linspace(1, 0, ramp)
            murmur_signal = murmur_intensity * envelope * (0.5 * broadband + 0.5 * tonal)
            signal[murmur_mask] += murmur_signal

        # --- S2 (chiusura aortica/polmonare) ---
        s2_center = s1_center + rr_interval * 0.45
        if s2_center >= duration:
            continue
        s2_amp  = 0.7 + np.random.uniform(-0.05, 0.05)
        s2_dur  = 0.06
        s2_freq = 120
        s2_mask = (t >= s2_center) & (t < s2_center + s2_dur)
        if s2_mask.sum() > 0:
            env_s2  = np.exp(-((t[s2_mask] - s2_center) / (s2_dur / 3)) ** 2)
            signal[s2_mask] += s2_amp * env_s2 * np.sin(2 * np.pi * s2_freq * t[s2_mask])

    # --- Rumore ambientale (simula condizioni di campo) ---
    signal += noise_level * np.random.randn(n_samples)

    # --- Normalizzazione ---
    signal = signal / (np.max(np.abs(signal)) + 1e-8)

    return signal.astype(np.float32)


def pcg_to_melspec(signal, sr=SR, n_mels=N_MELS, n_fft=N_FFT,
                   hop_length=HOP_LENGTH, sg_maxlen=SG_MAXLEN):
    """Converte un segnale PCG in log-Mel spectrogram normalizzato."""
    S    = librosa.feature.melspectrogram(
               y=signal, sr=sr, n_mels=n_mels,
               n_fft=n_fft, hop_length=hop_length
           )
    S_dB = librosa.power_to_db(S, ref=np.max)
    # Porta alla lunghezza standard
    if S_dB.shape[1] < sg_maxlen:
        pad = sg_maxlen - S_dB.shape[1]
        S_dB = np.pad(S_dB, ((0, 0), (0, pad)),
                      mode='constant', constant_values=S_dB.min())
    else:
        S_dB = S_dB[:, :sg_maxlen]
    return S_dB.astype(np.float32)


# Genera N campioni sintetici RHD Advanced
N_SYNTHETIC = 500  # puoi aumentare

print(f"Generazione di {N_SYNTHETIC} PCG sintetici RHD Advanced...")
X_synth_list = []
for i in tqdm(range(N_SYNTHETIC)):
    sig = generate_rhd_pcg(
        sr              = SR,
        duration        = 3.0,
        heart_rate      = np.random.randint(75, 110),   # variabilità HR pediatrica
        murmur_intensity= np.random.uniform(0.5, 0.8),  # severità variabile
        noise_level     = np.random.uniform(0.02, 0.08),# rumore ambientale
        seed            = i
    )
    sg = pcg_to_melspec(sig)
    X_synth_list.append(sg)

X_synth = np.array(X_synth_list, dtype=np.float32)
y_synth = np.full(N_SYNTHETIC, fill_value=2, dtype=np.int32)  # classe 2 = RHD Advanced

print(f"Sintetici generati — shape: {X_synth.shape}")

# Visualizza un esempio
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ax[0].plot(generate_rhd_pcg(seed=0))
ax[0].set_title("PCG sintetico RHD Advanced (dominio tempo)")
ax[0].set_xlabel("Campioni")
librosa.display.specshow(X_synth[0], sr=SR, hop_length=HOP_LENGTH,
                         x_axis='time', y_axis='mel', ax=ax[1])
ax[1].set_title("Log-Mel Spectrogram")
plt.tight_layout()
plt.show()

In [ ]:
# =====================================================================
# CELLA 5 — Unione dataset e split train/val/test
# =====================================================================

# Carica CirCor se già salvato (salta il reprocessing)
with open('/content/circor_data.pkl', 'rb') as f:
    circor = pickle.load(f)
X_circor = circor['X']
y_circor = circor['y']

# Unisci CirCor + sintetici
X_all = np.concatenate([X_circor, X_synth], axis=0)
y_all = np.concatenate([y_circor, y_synth], axis=0)

print(f"Dataset completo — shape X: {X_all.shape}, shape y: {y_all.shape}")
print(f"Distribuzione classi:")
for cls, name in enumerate(['Normal', 'Murmur/Subclinical RHD', 'RHD Advanced']):
    count = np.sum(y_all == cls)
    print(f"  Classe {cls} ({name}): {count} campioni")

# Aggiungi dimensione canale per CNN: (N, n_mels, sg_maxlen, 1)
X_all = X_all[..., np.newaxis]

# Normalizzazione globale min-max in [-1, 1]
X_min = X_all.min()
X_max = X_all.max()
X_all = 2 * (X_all - X_min) / (X_max - X_min + 1e-8) - 1

# Split stratificato: 70% train, 15% val, 15% test
X_train, X_temp, y_train, y_temp = train_test_split(
    X_all, y_all, test_size=0.30,
    stratify=y_all, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50,
    stratify=y_temp, random_state=42
)

print(f"\nTrain: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# Class weights per gestire lo sbilanciamento
cw = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)
class_weights = dict(enumerate(cw))
print(f"\nClass weights: {class_weights}")

In [ ]:
# =====================================================================
# CELLA 6 — Modello CNN + LSTM
# =====================================================================

def build_cnn_lstm(input_shape, n_classes=3, lstm_units=64, dropout=0.3):
    """
    Architettura:
      - CNN leggera (ispirata a MobileNet con depthwise separable conv)
        estrae feature spaziali dallo spettrogramma
      - Reshape per passare sequenza temporale all'LSTM
      - LSTM cattura dipendenze temporali tra frame successivi
      - Dense softmax per classificazione 3 classi
    """
    inp = layers.Input(shape=input_shape)  # (n_mels, sg_maxlen, 1)

    # --- Blocco CNN ---
    x = layers.Conv2D(32, (3, 3), padding='same', activation='relu')(inp)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(dropout)(x)

    x = layers.DepthwiseConv2D((3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(64, (1, 1), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 2))(x)
    x = layers.Dropout(dropout)(x)

    x = layers.DepthwiseConv2D((3, 3), padding='same', activation='relu')(x)
    x = layers.Conv2D(128, (1, 1), padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling2D((2, 1))(x)  # pool solo su asse mel, non tempo
    x = layers.Dropout(dropout)(x)

    # --- Reshape per LSTM ---
    # Dopo CNN: (batch, n_mels_reduced, time_steps, channels)
    # Vogliamo: (batch, time_steps, features) per LSTM
    cnn_shape = x.shape  # (None, h, w, c)
    x = layers.Permute((2, 1, 3))(x)                     # (None, w, h, c)
    x = layers.Reshape((cnn_shape[2], cnn_shape[1] * cnn_shape[3]))(x)

    # --- Blocco LSTM ---
    x = layers.LSTM(lstm_units, return_sequences=True)(x)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(lstm_units // 2)(x)
    x = layers.Dropout(dropout)(x)

    # --- Output ---
    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(dropout)(x)
    out = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs=inp, outputs=out, name='CNN_LSTM_RHD')
    return model


input_shape = X_train.shape[1:]  # (n_mels, sg_maxlen, 1)
model = build_cnn_lstm(input_shape, n_classes=3)
model.summary()

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.SparseTopKCategoricalAccuracy(k=2, name='top2_acc')]
)

In [ ]:
# =====================================================================
# CELLA 7 — Training
# =====================================================================

EPOCHS     = 50
BATCH_SIZE = 32

cb_list = [
    # Salva il modello migliore in base alla val_loss
    callbacks.ModelCheckpoint(
        '/content/best_model.keras',
        monitor='val_loss', save_best_only=True, verbose=1
    ),
    # Riduce il learning rate se val_loss non migliora per 5 epoche
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=5, min_lr=1e-6, verbose=1
    ),
    # Early stopping se val_loss non migliora per 12 epoche
    callbacks.EarlyStopping(
        monitor='val_loss', patience=12,
        restore_best_weights=True, verbose=1
    )
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weights,
    callbacks=cb_list
)
print("Training completato.")

In [ ]:
# =====================================================================
# CELLA 8 — Valutazione e grafici
# =====================================================================
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

# --- Curve di loss e accuracy ---
fig, axs = plt.subplots(1, 2, figsize=(14, 4))
axs[0].plot(history.history['loss'],     label='Train loss')
axs[0].plot(history.history['val_loss'], label='Val loss')
axs[0].set_title('Loss')
axs[0].legend()

axs[1].plot(history.history['accuracy'],     label='Train acc')
axs[1].plot(history.history['val_accuracy'], label='Val acc')
axs[1].set_title('Accuracy')
axs[1].legend()
plt.tight_layout()
plt.show()

# --- Valutazione sul test set ---
y_pred_prob = model.predict(X_test)
y_pred      = np.argmax(y_pred_prob, axis=1)

class_names = ['Normal', 'Murmur/Subclinical RHD', 'RHD Advanced']
print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# --- Confusion matrix ---
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names, ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('True')
ax.set_title('Confusion Matrix — Test Set')
plt.tight_layout()
plt.show()

# --- Metriche chiave per il report ---
from sklearn.metrics import recall_score, precision_score, f1_score
sensitivity = recall_score(y_test, y_pred, average='macro')
specificity_per_class = []
for i in range(len(class_names)):
    tn = cm.sum() - (cm[i, :].sum() + cm[:, i].sum() - cm[i, i])
    fp = cm[:, i].sum() - cm[i, i]
    specificity_per_class.append(tn / (tn + fp + 1e-8))

print(f"\nSensitivity (macro):  {sensitivity:.3f}")
print(f"Specificity per classe: {dict(zip(class_names, [f'{s:.3f}' for s in specificity_per_class]))}")
print(f"Must-have target: Sensitivity >= 0.85 | {'✅ PASS' if sensitivity >= 0.85 else '❌ NON RAGGIUNTO'}")

In [ ]:
# =====================================================================
# CELLA 9 — Conversione TFLite (Edge AI per Android)
# =====================================================================

# Carica il miglior modello salvato
best_model = tf.keras.models.load_model('/content/best_model.keras')

# Converti in TFLite con quantizzazione INT8
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]  # quantizzazione post-training

# Dataset rappresentativo per calibrazione INT8
def representative_dataset():
    for i in range(min(200, len(X_train))):
        yield [X_train[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type  = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

tflite_path = '/content/rhd_model_int8.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

size_fp32   = os.path.getsize('/content/best_model.keras') / 1e6
size_tflite = os.path.getsize(tflite_path) / 1e6
print(f"Modello Keras (.keras): {size_fp32:.2f} MB")
print(f"Modello TFLite INT8:    {size_tflite:.2f} MB")
print(f"Riduzione dimensione:   {(1 - size_tflite/size_fp32)*100:.1f}%")
print(f"\nFile TFLite salvato in: {tflite_path}")
print("Pronto per il deployment su Android via TensorFlow Lite.")